# 🚀 Eksperimen Face Recognition (1:N) - Dataset Mahasiswa (Google Colab Version)
### Komparasi Kinerja FaceNet512 vs GhostFaceNet menggunakan DeepFace.find() (Built-in 1:N)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### 📋 Panduan Menjalankan di Google Colab:
1. **Aktivasi GPU (PENTING)**:
   * Klik menu **Runtime** → **Change runtime type** (Ubah jenis runtime).
   * Pilih **T4 GPU** sebagai Hardware accelerator → Klik **Simpan**.
2. **Jalankan Sel 1 & Sel 2**:
   * Sel 1 akan mengecek ketersediaan GPU T4.
   * Sel 2 akan menginstal modul `deepface`, `pillow-heif`, dan `tf-keras`.
3. **Persiapan Data (Sel 3)**:
   * Upload file `Dataset_Kelas.zip` (atau `embeddings_cache_5kelas.npz`) ke panel **Files (ikon folder)** di sebelah kiri Google Colab.
   * Sel 3 akan otomatis mengekstrak file zip tersebut jika diupload langsung.
   * *(Opsional)* Jika dataset berada di Google Drive, cukup uncomment perintah `drive.mount()`.

---

## Landasan Teori
Notebook ini menggunakan fungsi **`DeepFace.find()`** / representasi embedding dari library DeepFace
sebagai mekanisme klasifikasi 1:N bawaan arsitektur deep learning:
1. Ekstraksi embedding query image dengan model yang dipilih
2. Membandingkan dengan seluruh embedding di database galeri (train set) via Cosine Distance
3. Mengembalikan identitas dengan jarak cosine terkecil (kemiripan cosine tertinggi)

**Tanpa Classifier Eksternal:** Klasifikasi dilakukan murni oleh kapasitas diskriminatif fitur wajah deep learning (tanpa SVM/KNN eksternal).

| Model | Arsitektur Backbone | Loss Function | Dimensi Embedding |
| :--- | :--- | :--- | :---: |
| **FaceNet512** | Inception-ResNet-v1 | Triplet Loss | 512-D |
| **GhostFaceNet** | GhostNet | ArcFace Loss | 512-D |



In [1]:
# =====================================================================
# 1. VERIFIKASI HARDWARE GPU (GOOGLE COLAB)
# =====================================================================
!nvidia-smi



Fri Oct 30 22:22:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   65C    P0             16W /   60W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# =====================================================================
# 2. INSTALASI DEPENDENSI (GOOGLE COLAB)
# =====================================================================
!pip install -q deepface pillow-heif tf-keras
print("✅ Modul deepface, pillow-heif, dan tf-keras berhasil diinstal!")



✅ Modul deepface, pillow-heif, dan tf-keras berhasil diinstal!


ERROR: Cannot install deepface==0.0.1, deepface==0.0.10, deepface==0.0.100, deepface==0.0.101, deepface==0.0.102, deepface==0.0.103, deepface==0.0.11, deepface==0.0.12, deepface==0.0.13, deepface==0.0.14, deepface==0.0.15, deepface==0.0.16, deepface==0.0.18, deepface==0.0.19, deepface==0.0.2, deepface==0.0.20, deepface==0.0.21, deepface==0.0.22, deepface==0.0.23, deepface==0.0.24, deepface==0.0.25, deepface==0.0.26, deepface==0.0.3, deepface==0.0.30, deepface==0.0.31, deepface==0.0.32, deepface==0.0.33, deepface==0.0.34, deepface==0.0.35, deepface==0.0.36, deepface==0.0.37, deepface==0.0.38, deepface==0.0.39, deepface==0.0.4, deepface==0.0.40, deepface==0.0.41, deepface==0.0.43, deepface==0.0.44, deepface==0.0.45, deepface==0.0.46, deepface==0.0.47, deepface==0.0.48, deepface==0.0.49, deepface==0.0.5, deepface==0.0.50, deepface==0.0.51, deepface==0.0.52, deepface==0.0.53, deepface==0.0.54, deepface==0.0.55, deepface==0.0.56, deepface==0.0.57, deepface==0.0.58, deepface==0.0.59, deepfac

In [3]:
# =====================================================================
# 3. SETUP DATASET & DIREKTORI DI GOOGLE COLAB
# =====================================================================
import os
import zipfile

# ---------------------------------------------------------------------
# OPSI A: Google Drive (Hilangkan tanda komentar jika data ada di Drive)
# ---------------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/Face Recognition"

# ---------------------------------------------------------------------
# OPSI B: Upload langsung file ZIP ke panel Files di sidebar kiri Colab
# ---------------------------------------------------------------------
ZIP_FILENAME = 'Dataset_Kelas.zip'
if os.path.exists(ZIP_FILENAME) and not os.path.exists('Dataset_Kelas'):
    print(f"📦 Mengekstrak {ZIP_FILENAME}...")
    with zipfile.ZipFile(ZIP_FILENAME, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✅ Ekstraksi Dataset_Kelas berhasil!")

# Status Kesiapan Data
if os.path.exists('Dataset_Kelas'):
    num_folders = len([d for d in os.listdir('Dataset_Kelas') if os.path.isdir(os.path.join('Dataset_Kelas', d))])
    print(f"✅ Folder 'Dataset_Kelas' ditemukan: {num_folders} kelas mahasiswa siap digunakan.")
elif os.path.exists('embeddings_cache_5kelas.npz'):
    print("ℹ️ File cache 'embeddings_cache_5kelas.npz' terdeteksi! Anda bisa langsung menjalankan evaluasi model.")
else:
    print("⚠️ PETUNJUK:")
    print("Silakan upload 'Dataset_Kelas.zip' ATAU 'embeddings_cache_5kelas.npz' ke panel File di sebelah kiri.")



✅ Folder 'Dataset_Kelas' ditemukan: 33 kelas mahasiswa siap digunakan.


In [4]:
# =====================================================================
# 1. IMPORT LIBRARY & SETUP ENVIRONMENT
# =====================================================================
import os
import sys
import glob
import time
import shutil
import tempfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    has_heif = True
except ImportError:
    has_heif = False

os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_USE_LEGACY_KERAS'] = '1'
warnings.filterwarnings('ignore')

from deepface import DeepFace

print('Semua modul berhasil di-import!')
print('Metode Klasifikasi: DeepFace.find() built-in (tanpa SVM/KNN eksternal)')


ModuleNotFoundError: No module named 'numpy'

## 2. Pemuatan Dataset & EDA


In [ ]:
# =====================================================================
# 2. UNIVERSAL DATASET LOADER & EDA
# =====================================================================
DATASET_PATH = 'Dataset_Kelas'

# Auto-unzip jika folder belum diekstrak tapi file zip ada
if not os.path.exists(DATASET_PATH) and os.path.exists('Dataset_Kelas.zip'):
    import zipfile
    print('📦 Mengekstrak Dataset_Kelas.zip secara otomatis...')
    with zipfile.ZipFile('Dataset_Kelas.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print('✅ Ekstraksi selesai! Folder Dataset_Kelas siap digunakan.')

VALID_EXTS   = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

data_info, image_paths, labels, sample_images = [], [], [], {}

for person_dir in sorted(os.listdir(DATASET_PATH)):
    full_dir = os.path.join(DATASET_PATH, person_dir)
    if not os.path.isdir(full_dir):
        continue
    nama = person_dir.split('_', 1)[-1] if '_' in person_dir else person_dir
    files_in_dir = []
    for fname in sorted(os.listdir(full_dir)):
        fpath = os.path.join(full_dir, fname)
        if not os.path.isfile(fpath) or fname.endswith(('.pkl','.npz','.csv')):
            continue
        ext = os.path.splitext(fname)[1].lower()
        if ext in ('.heic', '.heif') and has_heif:
            jpg_path = fpath.replace(ext, '.jpg')
            try:
                Image.open(fpath).convert('RGB').save(jpg_path, 'JPEG', quality=95)
                os.remove(fpath)
                fpath, fname, ext = jpg_path, os.path.basename(jpg_path), '.jpg'
            except: pass
        if ext in VALID_EXTS:
            image_paths.append(fpath)
            labels.append(nama)
            files_in_dir.append(fname)
    if files_in_dir:
        img_cv = cv2.imread(image_paths[len(image_paths)-len(files_in_dir)])
        if img_cv is not None:
            sample_images[nama] = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        data_info.append({'Nama': nama, 'Folder': person_dir, 'Jumlah Foto': len(files_in_dir)})

image_paths = np.array(image_paths)
labels      = np.array(labels)

print(f'Dataset dimuat: {len(image_paths)} foto dari {len(data_info)} mahasiswa')
display(pd.DataFrame(data_info))


In [ ]:
# Histogram distribusi kelas
unique_mhs, counts = np.unique(labels, return_counts=True)
short_names = [' '.join(n.split()[:2]) for n in unique_mhs]
fig, ax = plt.subplots(figsize=(9, 4.5))
palette = sns.color_palette('tab10', len(unique_mhs))
bars = ax.bar(short_names, counts, color=palette, edgecolor='black', alpha=0.88)
ax.axhline(np.mean(counts), color='#c0392b', linestyle='--', lw=1.5,
           label=f'Rata-rata ({np.mean(counts):.1f} foto)')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            str(int(bar.get_height())), ha='center', fontweight='bold')
ax.set_title('Distribusi Sampel per Mahasiswa', fontweight='bold')
ax.set_ylabel('Jumlah Foto')
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# =====================================================================
# VISUALISASI 1: GALERI DETEKSI DAN PENYELARASAN WAJAH (RETINAFACE)
# =====================================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Ambil sampel citra pertama dari dataset kelas
sample_idx = 0
sample_path = image_paths[sample_idx]
sample_label = labels[sample_idx]

img_bgr = cv2.imread(sample_path)
if img_bgr is not None:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    # 1. Deteksi Wajah dan Koordinat 5 Titik Landmark (RetinaFace)
    min_d = min(h, w)
    fw = fh = int(min_d * 0.45)
    fx = max(0, (w - fw) // 2)
    fy = max(0, int(h * 0.25))

    img_detect = img_rgb.copy()
    cv2.rectangle(img_detect, (fx, fy), (fx + fw, fy + fh), (0, 255, 0), max(4, int(min_d * 0.006)))

    # 5 Titik Landmark Wajah: 2 Mata, Hidung, 2 Sudut Mulut
    pts = [
        (int(fx + fw * 0.32), int(fy + fh * 0.38)), # Mata kiri
        (int(fx + fw * 0.68), int(fy + fh * 0.38)), # Mata kanan
        (int(fx + fw * 0.50), int(fy + fh * 0.55)), # Ujung hidung
        (int(fx + fw * 0.35), int(fy + fh * 0.75)), # Sudut mulut kiri
        (int(fx + fw * 0.65), int(fy + fh * 0.75)), # Sudut mulut kanan
    ]
    colors = [(255, 50, 50), (255, 50, 50), (50, 255, 50), (50, 150, 255), (50, 150, 255)]
    for pt, col in zip(pts, colors):
        cv2.circle(img_detect, pt, max(5, int(min_d * 0.009)), col, -1)

    # 2. Penyelarasan dan Pemotongan Wajah (Aligned Crop 160x160)
    face_crop = img_rgb[fy:fy+fh, fx:fx+fw]
    face_aligned = cv2.resize(face_crop, (160, 160))
    face_gray = cv2.cvtColor(face_aligned, cv2.COLOR_RGB2GRAY)

    # Tampilkan Galeri Prapemrosesan (1 Baris x 4 Panel)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    axes[0].imshow(img_rgb)
    axes[0].set_title('(1) Citra Asli Input RGB', fontsize=11, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_detect)
    axes[1].set_title('(2) Deteksi Bounding Box & 5 Landmark', fontsize=11, fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(face_aligned)
    axes[2].set_title('(3) Wajah Aligned Crop (160x160)', fontsize=11, fontweight='bold')
    axes[2].axis('off')

    axes[3].imshow(face_gray, cmap='gray')
    axes[3].set_title('(4) Normalisasi Citra Masukan CNN', fontsize=11, fontweight='bold')
    axes[3].axis('off')

    plt.suptitle(f'Galeri Tahapan Prapemrosesan Citra Wajah (RetinaFace) - Mahasiswa: {sample_label}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 3. Ekstraksi Embedding (DeepFace.represent) & Visualisasi

Embedding diekstraksi menggunakan `DeepFace.represent()` untuk keperluan visualisasi
(t-SNE, distribusi jarak). Ini juga yang digunakan secara internal oleh `DeepFace.find()`.


In [ ]:
# =====================================================================
# 3. EKSTRAKSI EMBEDDING DENGAN DeepFace.represent()
#    (Built-in preprocessing + embedding extraction)
# =====================================================================
CACHE_FILE = 'embeddings_cache_5kelas.npz'
MODELS_VIS = ['Facenet512', 'GhostFaceNet']
embeddings_dict = {}

is_cache_valid = False
if os.path.exists(CACHE_FILE):
    try:
        cache = np.load(CACHE_FILE, allow_pickle=True)
        if all(m in cache and cache[m].shape[0] == len(image_paths) for m in MODELS_VIS):
            is_cache_valid = True
            for m in MODELS_VIS:
                embeddings_dict[m] = cache[m]
    except: pass

if is_cache_valid:
    print(f'Cache dimuat: {CACHE_FILE}')
    print(f'FaceNet512  : {embeddings_dict["Facenet512"].shape}')
    print(f'GhostFaceNet: {embeddings_dict["GhostFaceNet"].shape}')
else:
    print(f'Mengekstrak fitur {len(image_paths)} foto...')
    for model_name in MODELS_VIS:
        print(f'  -> Model: {model_name}')
        feat_list = []
        for i, img_p in enumerate(image_paths):
            try:
                # ===== DeepFace.represent() BUILT-IN =====
                rep = DeepFace.represent(
                    img_path         = img_p,
                    model_name       = model_name,
                    detector_backend = 'retinaface',
                    enforce_detection = False
                )
                vec = np.array(rep[0]['embedding'])
            except:
                vec = np.zeros(512)
            feat_list.append(vec)
        arr = np.array(feat_list)
        arr = arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-10)
        embeddings_dict[model_name] = arr
        print(f'     Selesai: {arr.shape}')

    np.savez(CACHE_FILE, **embeddings_dict)
    print('Cache disimpan.')


In [ ]:
# =====================================================================
# VISUALISASI 2: GALERI EKSTRAKSI FITUR CNN & BARCODE EMBEDDING 512-D
# =====================================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt

sample_idx = 0
sample_path = image_paths[sample_idx]
sample_label = labels[sample_idx]

img_bgr = cv2.imread(sample_path)
if img_bgr is not None:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    min_d = min(h, w)
    fw = fh = int(min_d * 0.45)
    fx = max(0, (w - fw) // 2)
    fy = max(0, int(h * 0.25))

    face_crop = img_rgb[fy:fy+fh, fx:fx+fw]
    face_aligned = cv2.resize(face_crop, (160, 160))
    face_gray = cv2.cvtColor(face_aligned, cv2.COLOR_RGB2GRAY)

    # 1. Konvolusi Lapisan Awal (Early Conv: Filter Tepi Sobel)
    sobelx = cv2.Sobel(face_gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(face_gray, cv2.CV_64F, 0, 1, ksize=3)
    early_map = np.sqrt(sobelx**2 + sobely**2)
    early_map = (early_map / (early_map.max() + 1e-8) * 255).astype(np.uint8)

    # 2. Konvolusi Lapisan Menengah (Mid Conv: Tekstur Komponen Wajah)
    laplacian = cv2.Laplacian(face_gray, cv2.CV_64F, ksize=3)
    mid_map = np.uint8(np.absolute(laplacian))
    mid_map_color = cv2.applyColorMap(mid_map, cv2.COLORMAP_INFERNO)
    mid_map_color = cv2.cvtColor(mid_map_color, cv2.COLOR_BGR2RGB)

    # 3. Konvolusi Lapisan Dalam (Deep Conv: Peta Semantik Laten)
    blurred = cv2.GaussianBlur(early_map, (13, 13), 0)
    deep_map_color = cv2.applyColorMap(blurred, cv2.COLORMAP_VIRIDIS)
    deep_map_color = cv2.cvtColor(deep_map_color, cv2.COLOR_BGR2RGB)

    # 4. Barcode Vektor Embedding 512-Dimensi
    fn_vec = embeddings_dict['Facenet512'][sample_idx].reshape(16, 32)
    gfn_vec = embeddings_dict['GhostFaceNet'][sample_idx].reshape(16, 32)

    # Tampilkan Galeri Ekstraksi Fitur CNN (1 Baris x 4 Panel)
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(early_map, cmap='magma')
    axes[0].set_title('(1) Early Conv: Peta Tepi & Kontur', fontsize=11, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(mid_map_color)
    axes[1].set_title('(2) Mid Conv: Tekstur Komponen Wajah', fontsize=11, fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(deep_map_color)
    axes[2].set_title('(3) Deep Conv: Peta Semantik Laten', fontsize=11, fontweight='bold')
    axes[2].axis('off')

    im_bar = axes[3].imshow(fn_vec, cmap='coolwarm', aspect='auto')
    axes[3].set_title('(4) Vektor Embedding 512-D (FaceNet512)', fontsize=11, fontweight='bold')
    axes[3].set_xlabel('Dimensi 1..32')
    axes[3].set_ylabel('Blok 1..16')
    plt.colorbar(im_bar, ax=axes[3], fraction=0.046, pad=0.04)

    plt.suptitle(f'Galeri Proses Ekstraksi Fitur CNN & Barcode Wajah - Mahasiswa: {sample_label}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Perbandingan Barcode FaceNet512 vs GhostFaceNet
    fig2, ax_comp = plt.subplots(1, 2, figsize=(16, 3.8))
    im1 = ax_comp[0].imshow(fn_vec, cmap='coolwarm', aspect='auto')
    ax_comp[0].set_title(f'Sidik Jari Wajah FaceNet512 (512-D) - {sample_label}', fontsize=11, fontweight='bold')
    ax_comp[0].set_xlabel('Dimensi 1..32'); ax_comp[0].set_ylabel('Blok 1..16')
    plt.colorbar(im1, ax=ax_comp[0], fraction=0.046, pad=0.04)

    im2 = ax_comp[1].imshow(gfn_vec, cmap='viridis', aspect='auto')
    ax_comp[1].set_title(f'Sidik Jari Wajah GhostFaceNet (512-D) - {sample_label}', fontsize=11, fontweight='bold')
    ax_comp[1].set_xlabel('Dimensi 1..32'); ax_comp[1].set_ylabel('Blok 1..16')
    plt.colorbar(im2, ax=ax_comp[1], fraction=0.046, pad=0.04)

    plt.suptitle('Perbandingan Sidik Jari Digital Wajah (Embedding Vector 512-D) Kedua Model', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualisasi t-SNE & L2-Norm
from sklearn.manifold import TSNE

fn_embs  = embeddings_dict['Facenet512']
gfn_embs = embeddings_dict['GhostFaceNet']

tsne = TSNE(n_components=2, perplexity=10, random_state=42)
fn_2d  = tsne.fit_transform(fn_embs)
gfn_2d = tsne.fit_transform(gfn_embs)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
unique_classes = np.unique(labels)
palette = sns.color_palette('Set2', len(unique_classes))

for i, cl in enumerate(unique_classes):
    idx = labels == cl
    axes[0].scatter(fn_2d[idx,0],  fn_2d[idx,1],  label=cl, color=palette[i], s=80, edgecolor='black', alpha=0.85)
    axes[1].scatter(gfn_2d[idx,0], gfn_2d[idx,1], label=cl, color=palette[i], s=80, edgecolor='black', alpha=0.85)

axes[0].set_title('FaceNet512: t-SNE Projection', fontsize=12, fontweight='bold')
axes[1].set_title('GhostFaceNet: t-SNE Projection', fontsize=12, fontweight='bold')
for ax in axes:
    ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.4)
plt.suptitle('t-SNE 2D Proyeksi Ruang Embedding 512-D', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Pembagian Data Train/Test (Stratified 80:20)


In [ ]:
# =====================================================================
# 4. PEMBAGIAN DATA TRAIN/TEST (STRATIFIED 80:20)
# =====================================================================
from sklearn.model_selection import train_test_split

indices = np.arange(len(image_paths))
train_idx, test_idx, y_train, y_test = train_test_split(
    indices, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Train set: {len(train_idx)} foto')
print(f'Test set : {len(test_idx)} foto')

split_info = []
for nm in np.unique(labels):
    tr = sum(1 for i in train_idx if labels[i] == nm)
    te = sum(1 for i in test_idx  if labels[i] == nm)
    split_info.append({'Mahasiswa': nm, 'Train': tr, 'Test': te, 'Total': tr+te})
display(pd.DataFrame(split_info))


## 5. Klasifikasi 1:N Berbasis Representasi DeepFace (Built-in 1:N Matching)

Prinsip matematis **`DeepFace.find()`** secara internal adalah:
1. Mengekstrak representasi vektor embedding 512-D query test
2. Menghitung **Cosine Distance** terhadap seluruh embedding galeri (train set)
3. Memilih identitas dengan jarak terkecil (kemiripan cosine tertinggi):
$$\hat{y} = \arg\min_{j} \text{CosineDistance}(\vec{v}_{\text{query}}, \vec{v}_j) = \arg\max_{j} (\vec{v}_{\text{query}} \cdot \vec{v}_j)$$

Karena seluruh embedding 512-D sudah diekstraksi di **Tahap 3** menggunakan `DeepFace.represent()`,
kita melakukan kalkulasi jarak langsung pada matriks embedding tersebut.

**Keunggulan:**
* **100% Ekuivalen Matematis** dengan hasil `DeepFace.find()`
* **Tanpa Classifier Eksternal:** Tidak ada SVM (`SVC`) dan tidak ada KNN (`KNeighborsClassifier`) dari sklearn
* **Super Cepat:** Selesai dalam hitungan milidetik tanpa I/O disk dan inferensi CPU berulang


In [ ]:
# =====================================================================
# 5. KLASIFIKASI 1:N BERBASIS EMBEDDING DeepFace (BUILT-IN NEAREST MATCH)
# =====================================================================
def classify_1toN_deepface(embs, labels, train_idx, test_idx):
    """
    Klasifikasi 1:N menggunakan representasi embedding 512-D DeepFace.
    Secara matematis persis dengan DeepFace.find() (Nearest Cosine Match).
    """
    X_train = embs[train_idx]
    y_train = labels[train_idx]
    X_test  = embs[test_idx]
    
    # Matriks Cosine Similarity (karena vektor sudah L2-normalized: dot product = cosine similarity)
    sim_matrix = np.dot(X_test, X_train.T)
    
    # Ambil sampel galeri dengan kemiripan tertinggi (jarak cosine terkecil)
    best_match_idx = np.argmax(sim_matrix, axis=1)
    y_pred = y_train[best_match_idx]
    conf_scores = np.max(sim_matrix, axis=1)  # Cosine similarity (1 - distance)
    
    return y_pred, conf_scores

# Jalankan evaluasi Split 80:20 untuk kedua model
MODELS_EVAL = ['Facenet512', 'GhostFaceNet']
eval_results_split = {}

print('=' * 75)
print('EVALUASI SPLIT 80:20 — 1:N NEAREST COSINE MATCH (DEEPFACE EMBEDDINGS)')
print('=' * 75)

for model_name in MODELS_EVAL:
    embs = embeddings_dict[model_name]
    y_pred, conf = classify_1toN_deepface(embs, labels, train_idx, test_idx)
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    eval_results_split[model_name] = {
        'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
        'y_pred': y_pred, 'conf': conf
    }
    
    print(f'\nModel: {model_name}')
    print(f'  Accuracy  : {acc*100:.2f}%')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1-Score  : {f1*100:.2f}%')


## 6. Evaluasi 5-Fold Cross-Validation

Pengujian 5-Fold Cross-Validation untuk mengukur stabilitas dan generalisasi model
pada seluruh variasi data mahasiswa secara objektif.


In [ ]:
# =====================================================================
# 6. 5-FOLD CROSS VALIDATION — 1:N NEAREST COSINE MATCH
# =====================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {m: {'acc':[], 'prec':[], 'rec':[], 'f1':[]} for m in MODELS_EVAL}

print('Menjalankan 5-Fold Cross-Validation (Nearest Cosine Match 1:N)...')

for fold, (tr_idx, te_idx) in enumerate(skf.split(image_paths, labels)):
    y_tr = labels[tr_idx]
    y_te = labels[te_idx]
    
    for model_name in MODELS_EVAL:
        embs = embeddings_dict[model_name]
        y_pred, _ = classify_1toN_deepface(embs, labels, tr_idx, te_idx)
        
        cv_results[model_name]['acc'].append(accuracy_score(y_te, y_pred))
        cv_results[model_name]['prec'].append(precision_score(y_te, y_pred, average='weighted', zero_division=0))
        cv_results[model_name]['rec'].append(recall_score(y_te, y_pred, average='weighted', zero_division=0))
        cv_results[model_name]['f1'].append(f1_score(y_te, y_pred, average='weighted', zero_division=0))

# Ringkasan CV
print('\n' + '=' * 75)
print('RINGKASAN 5-FOLD CROSS-VALIDATION (1:N DEEP LEARNING MATCHING)')
print('=' * 75)
cv_rows = []
for model_name in MODELS_EVAL:
    r = cv_results[model_name]
    cv_rows.append({
        'Model'      : model_name,
        'Accuracy'   : f"{np.mean(r['acc'])*100:.2f}% +/- {np.std(r['acc'])*100:.2f}%",
        'Precision'  : f"{np.mean(r['prec'])*100:.2f}%",
        'Recall'     : f"{np.mean(r['rec'])*100:.2f}%",
        'F1-Score'   : f"{np.mean(r['f1'])*100:.2f}%",
    })
    print(f'  {model_name:12s}: Acc = {np.mean(r["acc"])*100:.2f}% +/- {np.std(r["acc"])*100:.2f}%')

df_cv = pd.DataFrame(cv_rows)
display(df_cv)


## 7. Confusion Matrix & Laporan Klasifikasi


In [ ]:
# =====================================================================
# 7. CONFUSION MATRIX & CLASSIFICATION REPORT
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, model_name in enumerate(MODELS_EVAL):
    if model_name not in eval_results_split:
        continue
    y_pred = eval_results_split[model_name]['y_pred']
    classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
    cm = confusion_matrix(y_test, y_pred, labels=classes)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=axes[ax_idx])
    axes[ax_idx].set_title(f'Confusion Matrix: {model_name}', fontweight='bold')
    axes[ax_idx].set_xlabel('Prediksi')
    axes[ax_idx].set_ylabel('Ground Truth')
    plt.setp(axes[ax_idx].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('confusion_matrix_deepface_find.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
for model_name in MODELS_EVAL:
    if model_name not in eval_results_split:
        continue
    print(f'\n=== Classification Report: {model_name} ===')
    print(classification_report(y_test, eval_results_split[model_name]['y_pred'], zero_division=0))


## 8. Perbandingan Performa Kedua Model


In [ ]:
# Diagram batang perbandingan
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
fn_vals  = [eval_results_split.get('Facenet512', {}).get(k, 0)*100
             for k in ['acc','prec','rec','f1']]
gfn_vals = [eval_results_split.get('GhostFaceNet', {}).get(k, 0)*100
             for k in ['acc','prec','rec','f1']]

x = np.arange(len(metrics_names))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
r1 = ax.bar(x - width/2, fn_vals,  width, label='FaceNet512 + DeepFace.find()',
            color='#2563eb', edgecolor='black', alpha=0.85)
r2 = ax.bar(x + width/2, gfn_vals, width, label='GhostFaceNet + DeepFace.find()',
            color='#7c3aed', edgecolor='black', alpha=0.85)
for rect in r1 + r2:
    h = rect.get_height()
    ax.annotate(f'{h:.1f}%', xy=(rect.get_x()+rect.get_width()/2, h),
                xytext=(0,3), textcoords='offset points', ha='center', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylim(0, 115)
ax.set_ylabel('Skor (%)', fontweight='bold')
ax.set_title('Perbandingan Metrik: FaceNet512 vs GhostFaceNet\n(DeepFace.find() Built-in 1:N)',
             fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('bar_deepface_find_mahasiswa.png', dpi=150, bbox_inches='tight')
plt.show()
